# CAPA Notebook

This notebook provides a working implementation for the Control-System Camera AI-Powered Productivity Assistant (CAPA). It includes a camera-based focus detector and real-time AI analysis using a local LLM API running at `http://127.0.0.1:1234`.


## Setup

Before running this notebook, ensure the local LLM API is running at `http://127.0.0.1:1234`.

Install the required Python packages, then run the next cells. The notebook will test the API connection and report success or failure.


In [19]:
!pip install transformers torch accelerate opencv-python numpy requests


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Load the LLM and define helper functions

This cell connects to the local LLM API running at `http://127.0.0.1:1234` and creates helper functions for generating productivity messages and session summaries. The API should be running before executing this cell.


In [20]:
import requests
import json

# Local LLM API endpoint
LOCAL_LLM_URL = "http://127.0.0.1:1234"
API_ENDPOINT = f"{LOCAL_LLM_URL}/v1/chat/completions"

def test_local_api():
    """Test connection to local LLM API"""
    try:
        response = requests.post(
            API_ENDPOINT,
            json={
                "model": "local",
                "messages": [{"role": "user", "content": "test"}],
                "max_tokens": 10,
                "temperature": 0.7
            },
            timeout=5
        )
        if response.status_code == 200:
            print("✓ Local LLM API connected successfully!")
            return True
        elif response.status_code == 400:
            error_data = response.json()
            if (
                "error" in error_data
                and isinstance(error_data["error"], dict)
                and "No models loaded" in str(error_data["error"].get("message", ""))
            ):
                print("⚠ Local LLM API connected, but no model is loaded!")
                print("   Please load a model in the developer page or use 'lms load' command.")
                print("   Visit: http://127.0.0.1:1234")
                return "no_model"
            else:
                print(f"✗ API returned error: {error_data}")
                return False
        else:
            print(f"✗ API returned status {response.status_code}")
            return False
    except Exception as e:
        print(f"✗ Failed to connect to local LLM API: {e}")
        return False

def generate_productivity_message(prompt: str, max_new_tokens: int = 128):
    """Generate productivity message using local LLM API"""
    try:
        response = requests.post(
            API_ENDPOINT,
            json={
                "model": "local",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_new_tokens,
                "temperature": 0.8,
                "top_p": 0.92,
            },
            timeout=30
        )

        raw_text = response.text
        try:
            data = response.json()
        except ValueError:
            data = None

        if response.status_code == 200:
            if isinstance(data, str):
                return data.strip()
            if isinstance(data, list) and len(data) > 0:
                first = data[0]
                if isinstance(first, str):
                    return first.strip()
                if isinstance(first, dict):
                    if "message" in first and isinstance(first["message"], dict) and "content" in first["message"]:
                        return first["message"]["content"].strip()
                    if "text" in first:
                        return str(first["text"]).strip()
            if isinstance(data, dict):
                if "choices" in data and isinstance(data["choices"], list) and len(data["choices"]) > 0:
                    choice = data["choices"][0]
                    if isinstance(choice, dict):
                        if "message" in choice and isinstance(choice["message"], dict) and "content" in choice["message"]:
                            return choice["message"]["content"].strip()
                        if "text" in choice:
                            return str(choice["text"]).strip()
                    elif isinstance(choice, str):
                        return choice.strip()
                if "output_text" in data:
                    return str(data["output_text"]).strip()
                if "text" in data:
                    return str(data["text"]).strip()
            if raw_text:
                return raw_text.strip()
            return "Unable to parse LLM response."
        elif response.status_code == 400:
            error_data = data if data is not None else raw_text
            if (
                isinstance(error_data, dict)
                and "error" in error_data
                and isinstance(error_data["error"], dict)
                and "No models loaded" in str(error_data["error"].get("message", ""))
            ):
                return "⚠ No model loaded in local LLM. Please load a model first at http://127.0.0.1:1234"
            return f"LLM returned error: {error_data}"

        return f"Unable to generate response from local LLM. Status: {response.status_code} Raw: {raw_text[:500]}"
    except Exception as e:
        return f"Error connecting to LLM: {str(e)}"


def summarize_session(events, max_events=20):
    """Generate session summary using local LLM API."""
    prompt_text = "You are an AI productivity coach. "
    prompt_text += "The user session data below describes desk presence, motion, and focus events. "
    prompt_text += "Create a concise summary outlining when the user was focused, when they were distracted, and one recommendation for improving the next session.\n\n"

    if len(events) > max_events:
        prompt_text += f"Only the last {max_events} events are included to fit the model context.\n\n"
        events_to_include = events[-max_events:]
    else:
        events_to_include = events

    prompt_text += "Session events:\n"
    for event in events_to_include:
        prompt_text += (
            f"- {event['timestamp']} | state={event['state']} | visible={event.get('visible', 'unknown')} | motion={int(event['motion'])}\n"
        )

    prompt_text += "\nSummary:"

    return generate_productivity_message(prompt_text, max_new_tokens=180)

print("Testing connection to local LLM API...")
api_connected = test_local_api()

if api_connected == "no_model":
    print("\n📋 To fix this:")
    print("1. Open http://127.0.0.1:1234 in your browser")
    print("2. Go to the 'Developer' page")
    print("3. Load a model (e.g., nvidia/nemotron-3-nano-4b or another available model)")
    print("4. Re-run this cell")
elif api_connected:
    print("✓ LLM helper functions configured for local API at 127.0.0.1:1234")
else:
    print("⚠ Warning: Local LLM API not found. Ensure it's running at http://127.0.0.1:1234")


Testing connection to local LLM API...
✓ Local LLM API connected successfully!
✓ LLM helper functions configured for local API at 127.0.0.1:1234


## Enhanced Camera Monitoring with Procrastination Detection

This enhanced version includes:
- **Real-time camera display** in a separate OpenCV window showing motion detection
- **Face and body awareness** so CAPA can tell if the user is truly present and engaged
- **Live transcript** logged directly in the notebook showing focus/quiet states
- **Procrastination detection** that identifies when motion is too low for extended periods
- **AI-powered suggestions** that provide personalized productivity tips when procrastination is detected
- **Focus analysis** with percentage calculations and recommendations at the end of the session

## Session States and Indicators

- **✓ FOCUSED**: Motion detected above threshold - user is actively engaged
- **⚠ QUIET**: Motion below threshold - user may be reading, thinking, or distracted
- **[ALERT]**: Procrastination detected - sustained low motion triggers AI suggestion
- **Camera Window**: Green border = focused, Orange border = quiet; Press 'Q' to exit

In [21]:
import cv2
import numpy as np
import time
from collections import deque
from IPython.display import clear_output, display
import sys

# Load face and body detectors
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

def detect_procrastination(motion_history, threshold_low=500, window_size=10):
    """
    Detect procrastination based on sustained low motion.
    Returns True if the user has been idle for too long.
    """
    if len(motion_history) < window_size:
        return False
    recent = list(motion_history)[-window_size:]
    avg_motion = np.mean(recent)
    return avg_motion < threshold_low


def detect_face_and_body(frame):
    """Detect faces and bodies in the current frame."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=6,
        minSize=(100, 100),
        flags=cv2.CASCADE_SCALE_IMAGE
    )

    resized = cv2.resize(frame, (640, int(frame.shape[0] * 640 / frame.shape[1])))
    gray_resized = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    bodies, _ = hog.detectMultiScale(gray_resized, winStride=(8, 8), padding=(8, 8), scale=1.05)

    # Scale body boxes back to original frame size
    scale = frame.shape[1] / 640
    body_boxes = [(int(x * scale), int(y * scale), int(w * scale), int(h * scale)) for (x, y, w, h) in bodies]

    return faces, body_boxes


def get_procrastination_suggestion(motion_level, idle_duration, visibility_desc):
    """
    Generate AI-powered suggestions when procrastination is detected.
    """
    if idle_duration < 5:
        return None
    
    prompt = (
        f"The user has been present but inactive for {idle_duration} seconds. "
        f"Current motion is low ({motion_level:.0f}) and visibility is '{visibility_desc}'. "
        "Generate a brief, encouraging productivity tip (1-2 sentences) to help them refocus."
    )
    
    suggestion = generate_productivity_message(prompt, max_new_tokens=64)
    return suggestion


def run_focus_session_enhanced(duration_seconds=60, min_motion_area=1500, display_interval=5):
    """
    Enhanced focus session with:
    - Real-time camera display in separate window
    - Session transcript logged to notebook
    - Face and body presence detection
    - Procrastination detection with AI suggestions
    """
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Unable to open the default webcam.")

    ret, frame = cap.read()
    if not ret:
        cap.release()
        raise RuntimeError("Unable to read from the webcam.")

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    background = cv2.GaussianBlur(gray, (21, 21), 0)
    events = []
    motion_history = deque(maxlen=30)
    face_history = deque(maxlen=15)
    idle_counter = 0
    last_display_time = time.time()
    start_time = time.time()
    transcript = []

    print("=" * 60)
    print("CAPA FOCUS SESSION STARTED")
    print("=" * 60)
    print(f"Duration: {duration_seconds} seconds")
    print(f"Motion threshold: {min_motion_area}")
    print("\nReal-time Transcript:")
    print("-" * 60)

    while time.time() - start_time < duration_seconds:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (21, 21), 0)
        diff = cv2.absdiff(background, blur)
        thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)[1]
        thresh = cv2.dilate(thresh, None, iterations=2)

        contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        motion_area = sum(cv2.contourArea(c) for c in contours)
        faces, bodies = detect_face_and_body(frame)
        face_present = len(faces) > 0
        body_present = len(bodies) > 0

        face_history.append(1 if face_present else 0)
        face_visibility_ratio = np.mean(face_history) if len(face_history) > 0 else 0.0
        stable_face = face_visibility_ratio >= 0.5

        if stable_face:
            user_visible = True
            visibility_desc = f"stable face seen ({face_visibility_ratio * 100:.0f}%)"
        elif face_present:
            user_visible = True
            visibility_desc = f"face detected ({len(faces)})"
        elif body_present:
            user_visible = True
            visibility_desc = f"body detected ({len(bodies)})"
        else:
            user_visible = False
            visibility_desc = "no user detected"

        if motion_area > min_motion_area and user_visible:
            state = "focused"
        elif user_visible:
            state = "quiet"
        else:
            state = "absent"

        motion_history.append(motion_area)
        timestamp = time.strftime("%H:%M:%S", time.localtime())
        elapsed = int(time.time() - start_time)

        event = {
            "timestamp": timestamp,
            "elapsed": elapsed,
            "motion": float(motion_area),
            "state": state,
            "visible": visibility_desc,
        }
        events.append(event)

        # Draw face/body boxes
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
        for (x, y, w, h) in bodies:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 0), 2)

        text = f"State: {state} | Motion: {motion_area:.0f} | {visibility_desc} | Time: {elapsed}s"
        cv2.putText(frame, text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        color = (0, 255, 0) if state == "focused" else (0, 165, 255) if state == "quiet" else (0, 0, 255)
        cv2.rectangle(frame, (10, 10), (frame.shape[1]-10, frame.shape[0]-10), color, 3)

        cv2.imshow("CAPA Focus Monitor", frame)

        is_procrastinating = detect_procrastination(motion_history) and state == "quiet"
        if is_procrastinating:
            idle_counter += 1
            if idle_counter % 10 == 0:
                suggestion = get_procrastination_suggestion(motion_area, idle_counter, visibility_desc)
                if suggestion:
                    msg = f"[ALERT {elapsed}s] Procrastination detected! Suggestion: {suggestion}"
                    transcript.append(msg)
                    print(msg)
        else:
            idle_counter = 0

        if time.time() - last_display_time >= display_interval:
            if state == "focused":
                log_msg = f"[{timestamp}] ✓ User is FOCUSED ({visibility_desc}, motion: {motion_area:.0f})"
            elif state == "quiet":
                log_msg = f"[{timestamp}] ⚠ User is QUIET ({visibility_desc}, motion: {motion_area:.0f})"
            else:
                log_msg = f"[{timestamp}] ⚠ User ABSENT ({visibility_desc})"
            transcript.append(log_msg)
            print(log_msg)
            last_display_time = time.time()

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\n[Session] Terminated by user (Q key pressed)")
            break

    cap.release()
    cv2.destroyAllWindows()
    print("-" * 60)
    print("CAPA FOCUS SESSION ENDED")
    print("=" * 60)
    return events, transcript

print("Enhanced camera focus session helper defined. Run `run_focus_session_enhanced()` to start monitoring.")

Enhanced camera focus session helper defined. Run `run_focus_session_enhanced()` to start monitoring.


## Run a Focus Session and Generate a Summary

Start the monitoring session and generate an AI summary from the collected events.

In [22]:
session_events, transcript = run_focus_session_enhanced(duration_seconds=60, min_motion_area=1500)

print("\n\nSESSION SUMMARY")
print("=" * 60)
print(f"Total events captured: {len(session_events)}")
print(f"Procrastination alerts: {sum(1 for t in transcript if 'ALERT' in t)}")

# Analyze focus patterns
focused_count = sum(1 for e in session_events if e["state"] == "focused")
quiet_count = len(session_events) - focused_count
focus_percentage = (focused_count / len(session_events) * 100) if session_events else 0

print(f"Focus time: {focus_percentage:.1f}%")
print(f"Quiet time: {100 - focus_percentage:.1f}%")

print("\nGenerating AI session analysis...")
summary = summarize_session(session_events)
print("\nAI-GENERATED SESSION SUMMARY:")
print("-" * 60)
print(summary)
print("-" * 60)

# Display final recommendation
if focus_percentage < 50:
    recommendation = "Your focus level was below average. Consider using the CAPA tool more frequently or adjusting your work environment to minimize distractions."
elif focus_percentage < 75:
    recommendation = "Good work! Your focus was above average. Keep maintaining consistent attention breaks and stay hydrated."
else:
    recommendation = "Excellent focus session! You maintained high concentration throughout. Keep up this momentum!"

print(f"\nRECOMMENDATION: {recommendation}")


CAPA FOCUS SESSION STARTED
Duration: 60 seconds
Motion threshold: 1500

Real-time Transcript:
------------------------------------------------------------
[16:43:06] ✓ User is FOCUSED (stable face seen (60%), motion: 212954)
[16:43:11] ✓ User is FOCUSED (body detected (1), motion: 185885)
[16:43:16] ⚠ User ABSENT (no user detected)
[16:43:21] ⚠ User ABSENT (no user detected)
[16:43:27] ⚠ User ABSENT (no user detected)
[16:43:32] ✓ User is FOCUSED (stable face seen (100%), motion: 222024)
[16:43:37] ✓ User is FOCUSED (stable face seen (100%), motion: 189917)
[16:43:42] ✓ User is FOCUSED (stable face seen (100%), motion: 168251)
[16:43:47] ✓ User is FOCUSED (stable face seen (100%), motion: 156800)
[16:43:52] ✓ User is FOCUSED (stable face seen (67%), motion: 135114)
[16:43:57] ✓ User is FOCUSED (stable face seen (100%), motion: 165382)
------------------------------------------------------------
CAPA FOCUS SESSION ENDED


SESSION SUMMARY
Total events captured: 964
Procrastination alerts